<a href="https://colab.research.google.com/github/Tribo310/MEARN/blob/main/nomio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="3WSlBQOLxQQbbDVYHJU2")
project = rf.workspace("nomin-g").project("final-tusul-fhl6y")
dataset = project.version(1).download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 67.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Final-tusul-1 in yolov8:: 100%|██████████| 8425/8425 [00:05<00:00, 1596.57it/s]


In [2]:
!pip install ultralytics
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.info()
results = model.train(data="/content/Final-tusul-1/data.yaml", epochs=3, imgsz=640)

results.results_dict

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Final-tusul-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epoc

{'metrics/precision(B)': 0.3259179109566113,
 'metrics/recall(B)': 0.3222835396197164,
 'metrics/mAP50(B)': 0.305201133507967,
 'metrics/mAP50-95(B)': 0.17620648954215284,
 'fitness': 0.17620648954215284}

In [6]:
!pip install supervision ultralytics
from ultralytics import YOLO
import supervision as sv
from tqdm import tqdm
import os

# =========================
# Video paths (Update these for Colab)
# =========================
# Upload your video to Colab and change this filename
SOURCE_VIDEO_PATH = "/content/tusul.mov"
TARGET_VIDEO_PATH = "/content/road_detection.mov"

# =========================
# Load trained YOLO model
# =========================
YOLO_MODEL_PATH = "/content/runs/detect/train/weights/best.pt"
if not os.path.exists(YOLO_MODEL_PATH):
    print(f"Warning: {YOLO_MODEL_PATH} not found. Ensure training finished.")
else:
    yolo_model = YOLO(YOLO_MODEL_PATH)

    # =========================
    # Annotators
    # =========================
    # Mapping colors to your 3 road classes
    colors = sv.ColorPalette.from_hex([
        "#00BFFF",  # evdrel (Class 0 or 1 depending on your yaml)
        "#FF0000",  # hagaral
        "#FFD700"   # tsuural
    ])

    box_annotator = sv.BoxAnnotator(color=colors, thickness=2)
    label_annotator = sv.LabelAnnotator(color=colors, text_thickness=2)

    # =========================
    # Video setup
    # =========================
    video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
    frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)

    # ByteTrack for temporal consistency
    tracker = sv.ByteTrack()

    with sv.VideoSink(TARGET_VIDEO_PATH, video_info=video_info) as video_sink:
        for frame in tqdm(frame_generator, desc="Processing Road Video"):
            # YOLO inference
            result = yolo_model.predict(frame, conf=0.25)[0]
            detections = sv.Detections.from_ultralytics(result)

            # Apply Non-Max Suppression
            detections = detections.with_nms(threshold=0.5)

            # Update tracker
            detections = tracker.update_with_detections(detections)

            # Map class IDs to names from model
            labels = [
                f"{yolo_model.model.names[class_id]} {conf:.2f}"
                for class_id, conf in zip(detections.class_id, detections.confidence)
            ]

            # Annotate frame
            annotated_frame = frame.copy()
            annotated_frame = box_annotator.annotate(
                scene=annotated_frame,
                detections=detections
            )
            annotated_frame = label_annotator.annotate(
                scene=annotated_frame,
                detections=detections,
                labels=labels
            )

            video_sink.write_frame(annotated_frame)

    print("\nRoad detection finished!")
    print(f"Saved video to: {TARGET_VIDEO_PATH}")

Processing Road Video: 0it [00:00, ?it/s]


0: 416x640 (no detections), 26.5ms
Speed: 6.0ms preprocess, 26.5ms inference, 3.1ms postprocess per image at shape (1, 3, 416, 640)


Processing Road Video: 1it [00:00,  2.78it/s]


0: 416x640 (no detections), 11.4ms
Speed: 5.8ms preprocess, 11.4ms inference, 0.9ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 (no detections), 12.8ms
Speed: 8.3ms preprocess, 12.8ms inference, 0.9ms postprocess per image at shape (1, 3, 416, 640)


Processing Road Video: 3it [00:00,  6.70it/s]

...


0: 416x640 (no detections), 22.1ms
Speed: 6.3ms preprocess, 22.1ms inference, 1.0ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 (no detections), 10.7ms
Speed: 3.9ms preprocess, 10.7ms inference, 1.0ms postprocess per image at shape (1, 3, 416, 640)


Processing Road Video: 2357it [02:46, 12.10it/s]


0: 416x640 (no detections), 9.6ms
Speed: 4.2ms preprocess, 9.6ms inference, 0.9ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 (no detections), 10.5ms
Speed: 5.0ms preprocess, 10.5ms inference, 0.8ms postprocess per image at shape (1, 3, 416, 640)


Processing Road Video: 2359it [02:46, 14.19it/s]


Road detection finished!
Saved video to: /content/road_detection.mov
